In [28]:
import pandas as pd
import numpy as np

In [29]:
data = pd.read_csv("cleaned_patents.csv")

# Remove the empty rows

In [30]:
data = data[data['clean_text'].astype(str).str.strip() != ""]
print(f"Rows remaining after removing empty data: {len(data)}")

Rows remaining after removing empty data: 2000


Defining the scoring function

In [31]:
def compute_genuine_novelty_score(text):
    if pd.isna(text):
        return 0
    
    text = str(text).lower()

    breakthrough_keywords = [
        'breakthrough', 'revolutionary', 'unprecedented', 'pioneer', 
        'fundamental', 'novelty', 'disruptive', 'paradigm shift'
    ]
    
    high_keywords = [
        'novel', 'inventive', 'unique', 'innovative', 'original', 
        'state-of-the-art', 'sophisticated', 'creative'
    ]
    
    mod_keywords = [
        'new', 'improved', 'enhanced', 'advanced', 'optimized', 
        'efficient', 'modern', 'superior', 'augmented'
    ]
    
    neg_keywords = [
        'conventional', 'prior art', 'standard', 'known', 'traditional', 
        'existing', 'previous', 'typical', 'ordinary', 'common'
    ]
    
    score = (sum(text.count(kw) for kw in breakthrough_keywords) * 5.0 +
                sum(text.count(kw) for kw in high_keywords) * 3.0 +
                sum(text.count(kw) for kw in mod_keywords) * 1.5 -
                sum(text.count(kw) for kw in neg_keywords) * 1.0)
    
    words = text.split()
    if len(words) > 0:
        diversity = len(set(words)) / len(words)
        score += (len(words) / 100) * diversity
        
    return score


# Applying the scoring function

In [32]:
data['raw_score'] = data['clean_text'].apply(compute_genuine_novelty_score)
data.columns

Index(['application_number', 'title', 'abstract', 'claims', 'text',
       'clean_text', 'raw_score'],
      dtype='object')

# Assign Tiers based on absolute thresholds

In [33]:
def assign_tier(score):
    if score >= 15:
        return 3
    elif score >= 5:
        return 2
    elif score >= 1:
        return 1
    else:
        return 0

# Applying the tiers

In [34]:
data['novelty tier'] = data['raw_score'].apply(assign_tier)

# Selecting only the required columns

In [35]:
labeled_data = data[['clean_text', 'novelty tier']]

create and save a data as CSV file

In [36]:
labeled_data.to_csv("labeled_patents.csv", index=False)
print(f"File saved successfully as {'labeled_patents.csv'}")
print("Tier Distribution:")
print(labeled_data['novelty tier'].value_counts().sort_index())

File saved successfully as labeled_patents.csv
Tier Distribution:
novelty tier
0     583
1    1258
2     107
3      52
Name: count, dtype: int64
